# NB01 – Real images + preprocess (v2)

Streams **5 K from COCO** and **5 K from ImageNet**, applies the canonical preprocess,
and writes `real/*.parquet`. No generation happens here.

- Internet **ON**, GPU **OFF** (CPU/IO bound — save your GPU quota).
- ImageNet terms accepted on your token's account.
- Resume-safe. Ends with a verifier.

In [8]:
import importlib.util, sys, subprocess
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
_need = []
for _m, _p in [("huggingface_hub","huggingface_hub>=0.23"),
               ("pyarrow","pyarrow"), ("PIL","pillow"), ("datasets","datasets")]:
    if importlib.util.find_spec(_m) is None: _need.append(_p)
if _need: _pip(*_need); print("installed:", _need)
else: print("all base deps present")

all base deps present


In [9]:
REPO_ID = "Shanmuk4622/ai-detection-dataset-v2"
import os
from huggingface_hub import hf_hub_download
def load_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t: return t.strip()
    except Exception: pass
    for k in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k): return os.environ[k].strip()
    import getpass; return getpass.getpass("HF write token: ").strip()
TOKEN = load_token()
lib = hf_hub_download(REPO_ID, "ai_detect_common.py", repo_type="dataset", token=TOKEN)
import importlib.util
spec = importlib.util.spec_from_file_location("ai_detect_common", lib)
C = importlib.util.module_from_spec(spec); spec.loader.exec_module(C)
cfg = C.read_config(REPO_ID, TOKEN)
MASTER_SEED = cfg.get("master_seed", 42)   # change master_seed in NB00 to reseed everything
print(f"library {C.PIPELINE_VERSION} loaded | MASTER_SEED={MASTER_SEED}")

ai_detect_common.py: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

library 1.2 loaded | MASTER_SEED=42


In [10]:
from datasets import load_dataset
import io as _io
from PIL import Image

api     = C.HfApi(token=TOKEN)
targets = cfg["real_sources"]
SOURCE_HF = cfg["real_source_ids"]

def _to_pil(v):
    if v is None: return None
    if isinstance(v, Image.Image): return v
    if isinstance(v, dict):
        if v.get("bytes"): return Image.open(_io.BytesIO(v["bytes"]))
        if v.get("path"):
            try: return Image.open(v["path"])
            except Exception: return None
    if isinstance(v, (bytes, bytearray)):
        return Image.open(_io.BytesIO(bytes(v)))
    return None

def _extract_image(ex):
    for key in ("image","img","jpg","png","image_data"):
        if key in ex:
            im = _to_pil(ex[key])
            if im is not None: return im
    for v in ex.values():
        im = _to_pil(v)
        if im is not None: return im
    return None

def iter_source(name):
    hfid = SOURCE_HF[name]
    ds   = load_dataset(hfid, split="train", streaming=True, token=TOKEN)
    for ex in ds:
        img = _extract_image(ex)
        if img is not None: yield img

# resume
existing = {s: 0 for s in targets}; max_idx = 0
for f in C.list_shards(REPO_ID, "real/", TOKEN):
    local = C.hf_hub_download(REPO_ID, f, repo_type="dataset", token=TOKEN)
    t = C.pq.read_table(local, columns=["source_dataset","image_id"])
    for s in t.column("source_dataset").to_pylist():
        if s in existing: existing[s] += 1
    for iid in t.column("image_id").to_pylist():
        try: max_idx = max(max_idx, int(str(iid).split("_")[-1]))
        except Exception: pass
gid = max_idx
print("existing:", existing, "| max id:", max_idx)

writer = C.ShardWriter(api, REPO_ID, "real/",
                       commit_interval=cfg["commit_interval_seconds"],
                       max_rows=cfg["batch_size"], token=TOKEN)
try:
    for name, want in targets.items():
        have = existing[name]
        if have >= want: print(f"{name}: complete ({have}/{want})"); continue
        need = want - have
        print(f"{name}: producing {need} more ...")
        produced = 0
        for img in iter_source(name):
            if produced >= need: break
            try:
                ow, oh = img.size
                png = C.canonical_preprocess(img)
            except Exception: continue
            gid += 1
            iid = f"real_{gid:06d}"
            row = C.empty_row()
            row.update(dict(image_id=iid, source_real_id=iid, label=0, generator="real",
                            source_dataset=name, prompt=None, image=png,
                            width=C.TARGET, height=C.TARGET,
                            orig_width=int(ow or 0), orig_height=int(oh or 0),
                            pipeline_version=C.PIPELINE_VERSION,
                            sha256=C.sha256_bytes(png), created_utc=C.now_utc()))
            writer.add(row); produced += 1
            if produced % 500 == 0: print(f"  {name}: {have+produced}/{want}")
            writer.maybe_flush()
        print(f"{name}: produced {produced} this run")
finally:
    writer.close()
print("real stage complete.")

existing: {'coco': 0, 'imagenet': 0} | max id: 0
coco: producing 5000 more ...


README.md:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

  committed 500 rows (9500 total) -> real/real-70beac4d-00018.parquet
  imagenet: 4500/5000


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  committed 500 rows (10000 total) -> real/real-70beac4d-00019.parquet
  imagenet: 5000/5000
imagenet: produced 5000 this run
real stage complete.


## Verifier

In [11]:
ok = C.verify_real_stage(REPO_ID, TOKEN, cfg)
print('verifier returned:', ok)

NB01 VERIFIER  –  real-image stage
real/ shards found: 20


real/real-70beac4d-00000.parquet:   0%|          | 0.00/222M [00:00<?, ?B/s]

real/real-70beac4d-00001.parquet:   0%|          | 0.00/225M [00:00<?, ?B/s]

real/real-70beac4d-00002.parquet:   0%|          | 0.00/222M [00:00<?, ?B/s]

real/real-70beac4d-00003.parquet:   0%|          | 0.00/222M [00:00<?, ?B/s]

...

real/real-70beac4d-00016.parquet:   0%|          | 0.00/203M [00:00<?, ?B/s]

real/real-70beac4d-00017.parquet:   0%|          | 0.00/208M [00:00<?, ?B/s]

real/real-70beac4d-00018.parquet:   0%|          | 0.00/204M [00:00<?, ?B/s]

real/real-70beac4d-00019.parquet:   0%|          | 0.00/203M [00:00<?, ?B/s]

total real rows: 10000
  coco: 5000 / 5000
  imagenet: 5000 / 5000
sampled 198 images: canonical_fail=0, sha_mismatch=0
----------------------------------------------------------------

RESULT: PASS  –  real stage looks correct.
verifier returned: True
